In [ ]:
# If you're in Colab, run this cell. If local, ensure these are installed in your venv.
!pip -q install transformers torch bertviz matplotlib umap-learn


Cell 2 — Imports & Setup

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from transformers import AutoTokenizer, AutoModel, AutoModelForMaskedLM
from bertviz import head_view

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Choose a small, common model
MODEL_NAME = "bert-base-uncased"

# Load encoder-only model for visualization
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
encoder_model = AutoModel.from_pretrained(MODEL_NAME)
encoder_model.eval()

# (Separate) MLM model for ablation experiments with logits
mlm_model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME)
mlm_model.eval()

print("Loaded:", MODEL_NAME)


## show_heads() helper (BertViz)

What it does:
Defines a function show_heads(sentence) that tries two BertViz calling conventions:

Modern API: head_view(encoder_model, tokenizer, sentence)

Legacy fallback: it manually runs a forward pass with output_attentions=True, then calls head_view(attentions, tokens).

Why it matters:
Different environments can have slightly different bertviz versions. This function makes the notebook robust to those differences, so your interactive attention widget appears reliably.

What to look for in the output:
An interactive visualization where you can select layers/heads and hover to see which tokens attend to which others—e.g., check how “it” attends to “animal” in pronoun-resolution examples.


In [ ]:
def show_heads(sentence: str):
    """
    Works with both bertviz API styles:
    - Newer: head_view(model, tokenizer, sentence)
    - Legacy: head_view(attentions, tokens)
    """
    try:
        # Newer API
        return head_view(encoder_model, tokenizer, sentence)
    except TypeError:
        # Legacy API path: compute attentions & tokens, then pass directly
        with torch.no_grad():
            inputs = tokenizer(sentence, return_tensors="pt")
            outputs = encoder_model(**inputs, output_attentions=True)
        attentions = outputs.attentions  # tuple: layers x (1, heads, seq, seq)
        tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
        return head_view(attentions, tokens)

# Quick smoke test (interactive widget will appear)
sentence_demo = "The animal didn't cross the street because it was too tired."
show_heads(sentence_demo)


Get Tokens & Attentions

In [ ]:
def get_tokens_and_attn(sentence: str):
    """
    Returns WordPiece tokens and a tuple of attention tensors per layer.
    attentions[layer] shape: (1, heads, seq, seq)
    """
    with torch.no_grad():
        inputs = tokenizer(sentence, return_tensors="pt")
        outputs = encoder_model(**inputs, output_attentions=True)
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    return tokens, outputs.attentions

# Test
tokens_test, attns_test = get_tokens_and_attn(sentence_demo)
print("Tokens:", tokens_test)
print("Layers:", len(attns_test), "Heads per layer:", attns_test[0].shape[1], "Seq len:", attns_test[0].shape[-1])


##Static Heatmap Helper
What it does:

Runs a sentence through the encoder to get attentions.

Selects one layer and one head (e.g., last layer, head 1).

Plots a (seq × seq) attention heatmap with tokens on both axes.

Why it matters:
This gives a fixed snapshot of how one head distributes attention from each query token to all keys—great for reports/screenshots.

What to look for:
Early layers often show local attention (to neighbors/punctuation). Later layers show more semantic structure (e.g., “it” → “animal”).

In [ ]:
def show_heatmap(attn_2d, tokens, title="Attention Heatmap"):
    fig = plt.figure(figsize=(6, 6))
    plt.imshow(attn_2d, interpolation="nearest")
    plt.xticks(range(len(tokens)), tokens, rotation=90)
    plt.yticks(range(len(tokens)), tokens)
    plt.title(title)
    plt.colorbar()
    plt.tight_layout()
    plt.show()

# Example: last layer, head 1
tokens, attentions = get_tokens_and_attn(sentence_demo)
num_layers = len(attentions)
num_heads = attentions[0].shape[1]

layer_idx = num_layers - 1
head_idx = 0

A = attentions[layer_idx][0, head_idx].cpu().numpy()  # (seq, seq)
show_heatmap(A, tokens, title=f"Layer {layer_idx+1}, Head {head_idx+1}")


Compare Early vs Late Layers (avg over heads)

What it does:

*  Averages attention over heads within a given layer.

*  Plots heatmaps for Layer 1 vs Last Layer.

Why it matters:
Averaging reduces head-specific noise to reveal layer-level behavior. You can compare how the model shifts from local patterns (early) to global/semantic patterns (late).

In [ ]:
def layer_avg(attentions, layer_idx: int):
    """
    Averages attention over heads for one layer.
    Returns (seq, seq) numpy array.
    """
    return attentions[layer_idx].mean(dim=1)[0].cpu().numpy()

early = layer_avg(attentions, 0)
late  = layer_avg(attentions, num_layers - 1)

show_heatmap(early, tokens, "Layer 1 (avg over heads)")
show_heatmap(late,  tokens, f"Layer {num_layers} (avg over heads)")


 Token-Centric Attention (top targets for a query token)

What it does:

* For a chosen token (e.g., "it"), extracts its attention distribution to all other tokens.

* Prints the top-k tokens that receive the most attention from that query token, for a chosen layer/head or averaged heads.

Why it matters:
This helps answer “who does ‘it’ refer to?” and whether attention captures coreference.

In [ ]:
def top_attended_tokens(attentions, tokens, query_token: str, layer_idx=None, head_idx=None, topk=5):
    if query_token not in tokens:
        print(f"Token {query_token!r} not found in tokens:\n{tokens}")
        return
    if layer_idx is None:
        layer_idx = len(attentions) - 1

    q_idx = tokens.index(query_token)
    if head_idx is None:
        A = attentions[layer_idx].mean(dim=1)[0]        # (seq, seq) avg heads
    else:
        A = attentions[layer_idx][0, head_idx]          # (seq, seq) one head

    dist = A[q_idx].cpu().numpy()
    order = np.argsort(dist)[::-1][:topk]
    print(f"Top-{topk} attention targets for {query_token!r} (Layer {layer_idx+1}"
          + (f", Head {head_idx+1}" if head_idx is not None else ", avg heads") + "):")
    for j in order:
        print(f"{tokens[j]:<12}  {dist[j]:.3f}")

# Example on demo sentence
top_attended_tokens(attentions, tokens, query_token="it", layer_idx=num_layers - 1, head_idx=None, topk=8)


Head Ablation with MLM (top-k at [MASK])

What it does:

*  Builds a masked sentence (e.g., replacing “ it ” with [MASK]).

*  Gets the baseline top-k predictions for the masked position using the MLM model.

*  Uses head_mask to drop (zero) one head in a chosen layer (commonly the last layer), then recomputes top-k predictions.

Why it matters:
 * If removing a head changes the top-k substantially, that head likely contributes to resolving the masked token—evidence of functional importance for that head.

How to interpret results:

* If “it” remains confidently predicted across ablations, the model may have redundant heads.

* If certain ablations make the predicted token shift (or confidence drop), those heads are salient for the task.

In [ ]:
def make_head_mask(num_layers: int, num_heads: int, drop=None):
    """
    drop = (layer_idx, head_idx) to zero a single head, else None.
    Returns head_mask with shape (num_layers, num_heads).
    """
    mask = torch.ones(num_layers, num_heads)
    if drop is not None:
        L, H = drop
        mask[L, H] = 0.0
    return mask

def topk_at_mask(text_with_mask: str, k=5, head_mask=None):
    inp = tokenizer(text_with_mask, return_tensors="pt")
    with torch.no_grad():
        out = mlm_model(**inp, head_mask=head_mask)
    logits = out.logits
    mask_pos = (inp["input_ids"][0] == tokenizer.mask_token_id).nonzero().item()
    top = torch.topk(logits[0, mask_pos], k=k)
    return [tokenizer.decode([i]) for i in top.indices]

# Build a masked sentence where "it" should be predicted
base = "The animal didn't cross the street because it was too tired."
masked = base.replace(" it ", f" {tokenizer.mask_token} ")

# Get sizes for head mask
with torch.no_grad():
    tmp = mlm_model(**tokenizer(masked, return_tensors="pt"), output_attentions=True)
mlm_layers = len(tmp.attentions)
mlm_heads  = tmp.attentions[0].shape[1]

print("Masked sentence:", masked)
baseline = topk_at_mask(masked, k=5, head_mask=make_head_mask(mlm_layers, mlm_heads, drop=None))
print("Baseline top-5 at [MASK]:", baseline)

# Ablate first few heads in the last layer; see how predictions shift
last_layer = mlm_layers - 1
for h in range(min(6, mlm_heads)):
    hm = make_head_mask(mlm_layers, mlm_heads, drop=(last_layer, h))
    preds = topk_at_mask(masked, k=5, head_mask=hm)
    print(f"Drop Layer {last_layer+1} Head {h+1}: {preds}")


A/B Probe Sentences (meaning shift)

In [ ]:
probe_sentences = [
    "The animal didn't cross the street because it was too tired.",
    "The animal didn't cross the street because it was too wide.",
    "I put the book on the shelf, then I picked it up again.",
    "Alice gave the violin to Bob because he was interested in music."
]

for s in probe_sentences:
    print("\n==>", s)
    toks, atts = get_tokens_and_attn(s)
    top_attended_tokens(atts, toks, query_token="it", layer_idx=len(atts)-1, head_idx=None, topk=8)


Embedding Space Peek (2D projection)

What it does:

* Uses output_hidden_states=True to access token embeddings at a chosen layer.

* Applies PCA to reduce embeddings to 2D.

* Plots tokens to visualize the geometry (clustering/spacing) at early vs late layers.

Why it matters:
Shows how token representations evolve across layers (from more lexical/positional to more semantic/task-oriented).

In [ ]:
from sklearn.decomposition import PCA

def plot_token_embeddings(sentence: str, layer_idx: int = 0):
    """
    Projects token embeddings (the hidden states for a layer) to 2D with PCA.
    Helpful to see clusters; not an attention map.
    """
    with torch.no_grad():
        inp = tokenizer(sentence, return_tensors="pt")
        out = encoder_model(**inp, output_hidden_states=True)
    hidden = out.hidden_states[layer_idx][0].cpu().numpy()  # (seq, hidden_dim)
    tokens = tokenizer.convert_ids_to_tokens(inp["input_ids"][0])

    z = PCA(n_components=2).fit_transform(hidden)
    fig = plt.figure(figsize=(6, 6))
    plt.scatter(z[:,0], z[:,1])
    for i, t in enumerate(tokens):
        plt.text(z[i,0], z[i,1], t)
    plt.title(f"Token embeddings — Layer {layer_idx+1}")
    plt.tight_layout()
    plt.show()

plot_token_embeddings(sentence_demo, layer_idx=0)   # early
plot_token_embeddings(sentence_demo, layer_idx=-1)  # late
